python scripts/reformat_dasa.py --datadir data --rename data/rename_columns.xlsx --correction data/fix_values.xlsx --output combined_dasa.tsv

In [1]:
import pandas as pd
import os

def load_table(file, separator=None):
    df = ''
    if str(file).split('.')[-1] == 'tsv':
        separator = '\t' if separator is None else separator
        df = pd.read_csv(file, encoding='latin-1', sep=separator, dtype='str')
    elif str(file).split('.')[-1] == 'csv':
        separator = ',' if separator is None else separator
        df = pd.read_csv(file, encoding='latin-1', sep=separator, dtype='str')
    elif str(file).split('.')[-1] in ['xls', 'xlsx']:
        df = pd.read_excel(file, index_col=None, header=0, sheet_name=0, dtype='str')
        df.fillna('', inplace=True)
    else:
        print('Wrong file format. Compatible file formats: TSV, CSV, XLS, XLSX')
        exit()
    return df

In [38]:
df_combined = pd.read_csv('../combined_dasa.tsv', sep='\t')

In [3]:
BASE_PATH = '../data/DASA/'
dfs = []

for file in os.listdir(BASE_PATH):
    if file.endswith(('.xlsx', '.tsv')):
        print(file)
        df = load_table(BASE_PATH + file, separator='\t')
        dfs.append(df)


codigo_dfs = [ df for df in dfs if 'codigo' in df.columns.tolist() ]
gene_s_dfs = []

for df in dfs:

    if 'Gene S' not in df.columns.tolist():
        continue

    rename_dict = {}
    df_columns = df.columns.tolist() 
    if 'resultado' not in df_columns:
        
        if 'resultado_norm' in df_columns:
            rename_dict['resultado_norm'] = 'resultado'
        elif 'resultado_original' in df_columns:
            # Nenhum caso encontrado
            rename_dict['resultado_original'] = 'resultado'

    if 'requisicao' not in df_columns:
        if 'codigo_externo_do_paciente' in df_columns:
            rename_dict['codigo_externo_do_paciente'] = 'requisicao'

    gene_s_dfs.append(df.rename(columns=rename_dict, errors='ignore'))

20220712_dados Dasa suspect_omicron_2022-07-03_2022-07-09-thermo para ITpS.xlsx
20230419_Dados Dasa patogenos_respiratorios_2023-04-09_2023-04-15 para ITpS_SE15.xlsx
20230713_Dados Dasa patogenos_respiratorios_2023-07-02_2023-07-08 para ITpS_SE27.xlsx
20220614_Dados Dasa patogenos_respiratorios_2022_06_05-2022_06_11 para ITpS.xlsx
20220621_Dados Dasa patogenos_respiratorios_2022-06-12_2022-06-18.xlsx
20221101_dados Dasa suspect_omicron_2022-10-09_2022-10-15-thermo para ITpS_SE41.xlsx
20220517_Dados Dasa patogenos_respiratorios_2022_05_08-2022_05_14 para ITpS.xlsx
20220722_Dados Dasa patogenos_respiratorios_2022-07-10_2022-07-16 para ITpS.xlsx
20220722_dados Dasa suspect_omicron_2022-07-10_2022-07-16-thermo para ITpS.xlsx
20221101_Dados Dasa suspect_omicron_2022-10-23_2022-10-29-thermo ITpS_SE43.xlsx
20220928_Dados Dasa patogenos_respiratorios_2022-09-18_2022-09-24 para ITpS.xlsx
20230529_Dados Dasa suspect_omicron_2023-05-14_2023-05-20-thermo para ITpS_SE20.xlsx
20221005_dados Dasa sus

In [4]:
gene_df = pd.concat(gene_s_dfs)
codigo_df = pd.concat(codigo_dfs)

## Duplicatas

In [27]:
df_duplicates = (
    df_combined
    .assign( one=1 )
    .groupby(['sample_id'])
    .agg(num_duplicates=('one', 'sum'))
    .query("num_duplicates > 1")
    .assign(num_duplicates=lambda x: x['num_duplicates']-1)
)


print( df_duplicates['num_duplicates'].sum(), df_duplicates['num_duplicates'].max(), df_duplicates['num_duplicates'].min() )
df_duplicates.sort_values('num_duplicates', ascending=False).head(10)

0 nan nan


,num_duplicates
sample_id,


In [28]:
df_duplicates_test_4 = (
    df_combined
    .query("test_kit == 'test_4'")
    .assign( one=1 )
    .groupby(['test_id'])
    .agg(num_duplicates=('one', 'sum'))
    .query("num_duplicates > 1")
    .assign(num_duplicates=lambda x: x['num_duplicates']-1)
)

df_duplicates_test_4

,num_duplicates
test_id,


In [29]:
df_duplicates = (
    df_combined
    # .query("test_kit == 'unknown'")
    .assign( one=1 )
    .groupby(['test_id'])
    .agg(num_duplicates=('one', 'sum'))
    .query("num_duplicates > 1")
    .assign(num_duplicates=lambda x: x['num_duplicates']-1)
)


print( df_duplicates['num_duplicates'].sum(), df_duplicates['num_duplicates'].max(), df_duplicates['num_duplicates'].min() )
df_duplicates.sort_values('num_duplicates', ascending=False).head(10)

0 nan nan


,num_duplicates
test_id,


In [30]:
df_os_multiple_tests = (
    df_combined
    .groupby('test_id')
    .agg(kits=('test_kit',lambda x: ','.join(x.tolist())))
    .query("kits.str.contains(',')", engine='python') 
)

In [31]:
df_os_multiple_tests.head(10)

,kits
test_id,


In [32]:
df_combined.columns

Index(['lab_id', 'test_id', 'test_kit', 'patient_id', 'sample_id', 'state',
       'location', 'date_testing', 'epiweek', 'age', 'sex', 'FLUA_test_result',
       'Ct_FluA', 'FLUB_test_result', 'Ct_FluB', 'VSR_test_result', 'Ct_VSR',
       'SC2_test_result', 'Ct_geneE', 'Ct_geneN', 'Ct_geneS', 'Ct_ORF1ab',
       'Ct_RDRP', 'geneS_detection', 'META_test_result', 'RINO_test_result',
       'PARA_test_result', 'ADENO_test_result', 'BOCA_test_result',
       'COVS_test_result', 'ENTERO_test_result', 'BAC_test_result'],
      dtype='object')

## Uniqueness

In [46]:
df_combined = pd.read_csv('../combined_dasa.tsv', sep='\t')

/tmp/ipykernel_6785/2677791513.py:1: DtypeWarning: Columns (1,10) have mixed types. Specify dtype option on import or set low_memory=False.
  df_combined = pd.read_csv('../combined_dasa.tsv', sep='\t')


In [47]:
if 'unknown' in df_combined['test_kit'].unique():
    print('WARNING! unknown test kits found')

df_combined['test_kit'].unique()

array(['thermo', 'test_4'], dtype=object)

In [48]:
df_combined['state'].unique()

array(['RIO DE JANEIRO', 'São Paulo', 'Santa Catarina', 'Pernambuco',
       'Distrito Federal', nan, 'Amazonas', 'Bahia', 'Espírito Santo',
       'Goiás', 'Minas Gerais', 'Mato Grosso do Sul', 'Mato Grosso',
       'Piauí', 'Paraná', 'Rio Grande do Sul', 'Pará', 'Ceará', 'Paraíba',
       'Rio Grande do Norte', 'Sergipe', 'Maranhão', 'Acre', 'Alagoas',
       'Amapá', 'Roraima', 'Rondônia', 'Tocantins'], dtype=object)

In [49]:
df_combined['location'].unique()

array(['MACAE', 'DIADEMA', 'SAO PAULO', ..., 'FORMIGA', 'PASSA QUATRO',
       'ABADIA'], dtype=object)

In [50]:
df_combined['date_testing'].max(), df_combined['date_testing'].min()

('2023-08-05', '2021-12-01')

In [51]:
df_combined['age'].unique(), df_combined['age'].min(), df_combined['age'].max()

(array([ 37,  23,  31,  56,  29,  44,  58,  49,  40,  13,   6,  25,  38,
         28,  36,  42,  24,  66,  41,  22,  50,  27,   1,  16,  53,  81,
         34,  59,  63,  39,  75,  32,  18,   3,  57,  20,  26,  19,  33,
         80,  45,  54,  73,  21,  60,  55,  35,  30,  43,  65,  51,  12,
         47,   9,  82,   5,  68,   0,  61,  89,  52,  64,  76,  62,  46,
         48,  11,  74,  77,  67,  14,  70,  72,  17,  69,  83,  10,   2,
         15,   8,  87,  71,  78,   4,  92,   7,  79,  85,  97,  88,  84,
         90,  86,  91,  94,  93,  96, 120,  95,  98, 100, 101,  99, 104,
         -1, 114, 102, 106, 105, 103, 107, 110, 117]),
 -1,
 120)

In [53]:
df_combined['sex'].unique()

array([nan, 'M', 'F'], dtype=object)

In [54]:
for column in df_combined.columns:
    if column.endswith('_result'):
        print(column, df_combined[column].unique())

FLUA_test_result ['NT' 'Pos' 'Neg']
FLUB_test_result ['NT' 'Neg' 'Pos']
VSR_test_result ['NT' 'Pos' 'Neg']
SC2_test_result ['Neg' 'Pos' 'NEGATIVO' 'POSITIVO (SEM CT)']
META_test_result ['NT']
RINO_test_result ['NT']
PARA_test_result ['NT']
ADENO_test_result ['NT']
BOCA_test_result ['NT']
COVS_test_result ['NT']
ENTERO_test_result ['NT']
BAC_test_result ['NT']


In [84]:
for col in df_combined.columns:
    if col.endswith('_result'):
        print(col)

FLUA_test_result
FLUB_test_result
VSR_test_result
SC2_test_result
META_test_result
RINO_test_result
PARA_test_result
ADENO_test_result
BOCA_test_result
COVS_test_result
ENTERO_test_result
BAC_test_result


## Inconsistencies

In [85]:
df_combined.columns

Index(['lab_id', 'test_id', 'test_kit', 'patient_id', 'sample_id', 'state',
       'location', 'date_testing', 'epiweek', 'age', 'sex', 'FLUA_test_result',
       'Ct_FluA', 'FLUB_test_result', 'Ct_FluB', 'VSR_test_result', 'Ct_VSR',
       'SC2_test_result', 'Ct_geneE', 'Ct_geneN', 'Ct_geneS', 'Ct_ORF1ab',
       'Ct_RDRP', 'geneS_detection', 'META_test_result', 'RINO_test_result',
       'PARA_test_result', 'ADENO_test_result', 'BOCA_test_result',
       'COVS_test_result', 'ENTERO_test_result', 'BAC_test_result'],
      dtype='object')

Verificando se um mesmo test_id + test_kit apresentam mais de um resultado para um mesmo patógeno

In [86]:
agg_test_rules = {
    col+'_kit': (col, lambda x: ','.join(x.tolist()))
    for col in df_combined.columns
    if col.endswith('_result')
}

df_os_non_contraditory_test_results = (
    df_combined
    .groupby(['test_id', 'test_kit'])
    .agg(**agg_test_rules)
)

In [94]:
for col in df_os_non_contraditory_test_results.columns:
    if col.endswith('_kit'):
        print(col, df_os_non_contraditory_test_results[col].unique())
        # Devem ser apenas ['NT' 'Neg' 'Pos']

FLUA_test_result_kit ['NT' 'NT,NT' 'NT,NT,NT' 'NT,NT,NT,NT' 'NT,NT,NT,NT,NT'
 'NT,NT,NT,NT,NT,NT,NT,NT,NT'
 'NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT' 'Neg' 'Neg,Neg'
 'NT,NT,NT,NT,NT,NT,NT' 'Pos' 'NT,NT,NT,NT,NT,NT,NT,NT,NT,NT' 'Pos,Pos'
 'NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT'
 'NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT']
FLUB_test_result_kit ['NT' 'NT,NT' 'NT,NT,NT' 'NT,NT,NT,NT' 'NT,NT,NT,NT,NT'
 'NT,NT,NT,NT,NT,NT,NT,NT,NT'
 'NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT' 'Neg' 'Neg,Neg'
 'NT,NT,NT,NT,NT,NT,NT' 'Pos' 'NT,NT,NT,NT,NT,NT,NT,NT,NT,NT' 'Pos,Pos'
 'NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT'
 'NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT']
VSR_test_result_kit ['NT' 'NT,NT' 'NT,NT,NT' 'NT,NT,NT,NT' 'NT,NT,NT,NT,NT'
 'NT,NT,NT,NT,NT,NT,NT,NT,NT'
 'NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT' 'Neg' 'Pos' 'Neg,Neg'
 'Pos,Pos' 'NT,NT,NT,NT,NT,NT,NT' 'NT,NT,NT,NT,NT,NT,NT,NT,NT,NT'
 'NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT'
 'NT,NT,NT,NT,NT,NT,NT,NT,NT,NT,NT' 'Neg,Pos']
SC2_test_result_kit ['NT'

In [95]:
df_os_non_contraditory_test_results.query("SC2_test_result_kit == 'Neg,Pos'")

,,FLUA_test_result_kit,FLUB_test_result_kit,VSR_test_result_kit,SC2_test_result_kit,META_test_result_kit,RINO_test_result_kit,PARA_test_result_kit,ADENO_test_result_kit,BOCA_test_result_kit,COVS_test_result_kit,ENTERO_test_result_kit,BAC_test_result_kit
test_id,test_kit,,,,,,,,,,,,
10007835,thermo,"NT,NT","NT,NT","NT,NT","Neg,Pos","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
1002751831,thermo,"NT,NT","NT,NT","NT,NT","Neg,Pos","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
1004143174,thermo,"NT,NT","NT,NT","NT,NT","Neg,Pos","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
1004249044,thermo,"NT,NT","NT,NT","NT,NT","Neg,Pos","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
11008254,thermo,"NT,NT","NT,NT","NT,NT","Neg,Pos","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
80118134,thermo,"NT,NT","NT,NT","NT,NT","Neg,Pos","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
92824455,thermo,"NT,NT","NT,NT","NT,NT","Neg,Pos","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
94009869,thermo,"NT,NT","NT,NT","NT,NT","Neg,Pos","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"


In [96]:
df_os_non_contraditory_test_results.reset_index().query("test_kit == 'test_4'").query("FLUA_test_result_kit == 'Neg,Neg'")

,test_id,test_kit,FLUA_test_result_kit,FLUB_test_result_kit,VSR_test_result_kit,SC2_test_result_kit,META_test_result_kit,RINO_test_result_kit,PARA_test_result_kit,ADENO_test_result_kit,BOCA_test_result_kit,COVS_test_result_kit,ENTERO_test_result_kit,BAC_test_result_kit
30639,279800084151,test_4,"Neg,Neg","Neg,Neg","Neg,Neg","Pos,Pos","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
30640,279800084175,test_4,"Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
30641,279800084182,test_4,"Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
30642,279800084199,test_4,"Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
30643,279800084229,test_4,"Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
385842,934401164311,test_4,"Neg,Neg","Neg,Neg","Pos,Pos","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
385843,934401165233,test_4,"Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
385844,934401166711,test_4,"Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"
385845,934401167060,test_4,"Neg,Neg","Neg,Neg","Neg,Neg","Neg,Neg","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT","NT,NT"


In [90]:
codigo_df.query("codigorequisicao == '279800084151'")

,ano_mes,data_exame,nome_pac,sexo,idade,cidade,uf,telefone,email,codigorequisicao,codigo,positivo
2800,2022-03,2022-03-07,NaN,F,43,NaN,DF,NaN,NaN,279800084151,COVID,1
2801,2022-03,2022-03-07,NaN,F,43,NaN,DF,NaN,NaN,279800084151,FLUA,0
2802,2022-03,2022-03-07,NaN,F,43,NaN,DF,NaN,NaN,279800084151,FLUB,0
2803,2022-03,2022-03-07,NaN,F,43,NaN,DF,NaN,NaN,279800084151,VSR,0
181,2022-03,2022-03-07,NaN,F,43,BRASILIA,DF,NaN,NaN,279800084151,FLUA,0
863,2022-03,2022-03-07,NaN,F,43,BRASILIA,DF,NaN,NaN,279800084151,FLUB,0
1545,2022-03,2022-03-07,NaN,F,43,BRASILIA,DF,NaN,NaN,279800084151,VSR,0
2227,2022-03,2022-03-07,NaN,F,43,BRASILIA,DF,NaN,NaN,279800084151,COVID,1


In [91]:
gene_df.query("requisicao == '1002751831'")

,requisicao,data,data_de_nascimento,idade,sexo,teste,data_do_resultado,data_de_liberacao,resultado_original,cidade_norm,...,apoio_cidade,gliese_uf,dasa_uf,apoio_uf,cep_municipio,cep_uf,cep_municipio_dasa,cep_uf_dasa,cidade_tratado,uf_tratado
6158,1002751831,2022-07-13 10:33:46+00:00,1984-02-03,38,FEMININO,DETECCAO QUALITATIVA DE CORONAVIRUS (2019-NCOV),2022-07-14 00:37:50+00:00,2022-07-14 01:13:35+00:00,NDT,SAOPAULO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
35,1002751831,2022-10-24 10:02:24+00:00,1984-02-03,38,FEMININO,DETECCAO QUALITATIVA DE CORONAVIRUS (2019-NCOV),2022-10-25 07:43:50+00:00,NaN,NaN,SAOPAULO,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [92]:
df_combined.query("test_id == '279800084151'")

,lab_id,test_id,test_kit,patient_id,sample_id,state,location,date_testing,epiweek,age,...,Ct_RDRP,geneS_detection,META_test_result,RINO_test_result,PARA_test_result,ADENO_test_result,BOCA_test_result,COVS_test_result,ENTERO_test_result,BAC_test_result
31584,DASA,279800084151,test_4,NaN,7b1859876e805cc6,Distrito Federal,NaN,2022-03-07,2022-03-12,43,...,NaN,NaN,NT,NT,NT,NT,NT,NT,NT,NT
31585,DASA,279800084151,test_4,NaN,dcb850bb21133ebf,Distrito Federal,BRASILIA,2022-03-07,2022-03-12,43,...,NaN,NaN,NT,NT,NT,NT,NT,NT,NT,NT


### Ids

Verificando se todos os Ids originais etão presentes no arquivo final

In [93]:
set_test_ids_final_df = set(df_combined['test_id'])
set_test_ids_raw_data = set(df_full_raw['CODIGO REQUISICAO'])

NameError: name 'df_full_raw' is not defined

In [ ]:
# Testar se todos os test_ids da planilha original estão na planilha final
# deve retornar um set vazio
set_test_ids_raw_data.difference(set_test_ids_final_df)

{'1200085178_100',
 '1410231927_100',
 '1421499943_100',
 '1421512238_100',
 '1421512252_100',
 '1421524508_100',
 '1421542791_100',
 '1421567484_600',
 '1421573025_900',
 '1421574120_100',
 '1421580321_100',
 '1421580345_100',
 '1421580359_600',
 '1421580372_100',
 '1421580414_700',
 '1421590360_100',
 '1421608991_700',
 '1421618661_100',
 '1421623648_100',
 '1421627267_100',
 '1421641149_200',
 '1421661054_100',
 '1421747497_400',
 '1450064023_100',
 '1450064862_100',
 '1600683873_100',
 '1600708233_100',
 '1600709104_100',
 '1600712587_100',
 '1650441468_100',
 '1650447361_100',
 '1650451652_100',
 '1660378198_100',
 '1720397744_100',
 '1860281386_100',
 '1870163611_100',
 '2230148082_100',
 '2240159856_100',
 '2240169073_100',
 '2240170521_100',
 '2240193197_100',
 '2240197205_100',
 '2310036024_100',
 '2320523931_100',
 '2320536507_100',
 '2320542723_100',
 '2320550391_100',
 '2320551123_100',
 '2320551273_100',
 '2320562210_100',
 '2320582241_100',
 '2350037816_100',
 '2350041791

In [ ]:
# Verificar se há test_ids na planilha final que não estão na planilha original
# deve retornar um set vazio
set_test_ids_final_df.difference(set_test_ids_raw_data)

set()

Verificando se todos os patógenos de um test_kit estão sendo testados para todos os test_id

In [ ]:
PATHOGENS_TESTS = {
    'panel_21':{
        # PAINCOVI
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'SC2_test_result',
        'META_test_result',
        'RINO_test_result',
        'PARA_test_result',
        'ADENO_test_result',
        'COVS_test_result',
        'BAC_test_result',
    },

    'panel_24':{
        # RESPIRA
        'FLUA_test_result',
        'FLUB_test_result',
        'VSR_test_result',
        'META_test_result',
        'RINO_test_result',
        'PARA_test_result',
        'ADENO_test_result',
        'BOCA_test_result',
        'COVS_test_result',
        'ENTERO_test_result',
        'BAC_test_result',
    },

    'flu_antigen':{
        'FLUA_test_result',
        'FLUB_test_result',
    },

    'covid_antigen':{
        'SC2_test_result',
    },

    'covid_pcr':{
        'SC2_test_result',
    },
}

In [ ]:
# Mapeando Pos e Neg para Tes
df_combined_test_pathogen_non_nt_on_kit = (
    df_combined
    .replace(('Pos', 'Neg'),  'Tes')
)

In [ ]:
for pathogen, test_columns in PATHOGENS_TESTS.items():
    print(pathogen)
    print(
        df_combined_test_pathogen_non_nt_on_kit
        .query("test_kit == @pathogen")
        [list(test_columns)]
        .drop_duplicates().T
        # Deve resultar em apenas uma linha completa por 'Tes'
    )

    print('\n\n')

panel_21
Empty DataFrame
Columns: []
Index: [FLUB_test_result, PARA_test_result, BAC_test_result, COVS_test_result, META_test_result, VSR_test_result, ADENO_test_result, RINO_test_result, FLUA_test_result, SC2_test_result]



panel_24
Empty DataFrame
Columns: []
Index: [FLUB_test_result, PARA_test_result, ENTERO_test_result, BAC_test_result, COVS_test_result, META_test_result, VSR_test_result, ADENO_test_result, BOCA_test_result, RINO_test_result, FLUA_test_result]



flu_antigen
Empty DataFrame
Columns: []
Index: [FLUB_test_result, FLUA_test_result]



covid_antigen
Empty DataFrame
Columns: []
Index: [SC2_test_result]



covid_pcr
                1929
SC2_test_result  Tes





Verificando resultados de alguns testes

In [ ]:
import numpy as np

In [ ]:
np.random.seed(214)

ids = np.random.choice(df_combined['test_id'], 100)
test_columns = [col for col in df_combined.columns if col.endswith('_result')]

In [ ]:
current_id=50

In [ ]:
current_id = current_id+1
id = ids[current_id]
print( df_combined.query("test_id == @id")[['test_id', 'test_kit'] + test_columns].T )
print( df_hla_covid.query(f"Pedido == '{id}'") [['Pedido'] + [col for col in df_hla_covid.columns if col.startswith('Vírus')]].T )
print( df_hla_ctn.query(f"Pedido == '{id}'") [['Pedido', 'Resultado']].T )
print( df_hla_flu.query(f"Pedido == '{id}'") [['Pedido']+ [col for col in df_hla_flu.columns if col.startswith('VIRUS') or col.startswith('BAC')]].T )
print( df_hla_ph4.query(f"Pedido == '{id}'") [['Pedido']+ [col for col in df_hla_ph4.columns if col.startswith('VIRUS') or col.startswith('BAC')]].T )

                         7665
test_id               2262865
test_kit            covid_pcr
FLUA_test_result           NT
FLUB_test_result           NT
VSR_test_result            NT
SC2_test_result           Neg
META_test_result           NT
RINO_test_result           NT
PARA_test_result           NT
ADENO_test_result          NT
BOCA_test_result           NT
COVS_test_result           NT
ENTERO_test_result         NT
BAC_test_result            NT
Empty DataFrame
Columns: []
Index: [Pedido, Vírus Influenza A, Vírus Influenza B, Vírus Sincicial Respiratório A/B]
                   1912
Pedido          2262865
Resultado  Inconclusivo
Empty DataFrame
Columns: []
Index: [Pedido, VIRUS_Influenza A, VIRUS_Influenza H1N1, VIRUS_Influenza H3, VIRUS_Influenza B, VIRUS_Metapneumovírus, VIRUS_Sincicial A, VIRUS_Sincicial B, VIRUS_Rinovírus, VIRUS_Parainfluenza 1, VIRUS_Parainfluenza 2, VIRUS_Parainfluenza 3, VIRUS_Parainfluenza 4, VIRUS_Adenovirus, VIRUS_Bocavirus, VIRUS_CoV-229E, VIRUS_CoV-HKU, VI

Verificando resultados de alguns testes - Fltrando por test_kit

In [ ]:
np.random.seed(214)

ids = np.random.choice(df_combined.query("test_kit not in ('covid_pcr', 'covid_antigen')")['test_id'], 100)
test_columns = [col for col in df_combined.columns if col.endswith('_result')]

In [ ]:
current_id=50

In [ ]:
current_id = current_id+1
id = ids[current_id]
print( df_combined.query("test_id == @id")[['test_id', 'test_kit'] + test_columns].T )
print( df_hla_covid.query(f"Pedido == '{id}'") [['Pedido'] + [col for col in df_hla_covid.columns if col.startswith(('Vírus', 'Coron'))]].T )
print( df_hla_flu.query(f"Pedido == '{id}'") [['Pedido']+ [col for col in df_hla_flu.columns if col.startswith('VIRUS') or col.startswith('BAC')]].T )
print( df_hla_ph4.query(f"Pedido == '{id}'") [['Pedido']+ [col for col in df_hla_ph4.columns if col.startswith('VIRUS') or col.startswith('BAC')]].T )

                      50943
test_id             2354190
test_kit             test_4
FLUA_test_result        Neg
FLUB_test_result        Neg
VSR_test_result         Pos
SC2_test_result         Neg
META_test_result         NT
RINO_test_result         NT
PARA_test_result         NT
ADENO_test_result        NT
BOCA_test_result         NT
COVS_test_result         NT
ENTERO_test_result       NT
BAC_test_result          NT
                                             18
Pedido                                  2354190
Vírus Influenza A                 Não Detectado
Vírus Influenza B                 Não Detectado
Vírus Sincicial Respiratório A/B      Detectado
Coronavírus SARS-CoV-2            Não Detectado
Empty DataFrame
Columns: []
Index: [Pedido, VIRUS_Influenza A, VIRUS_Influenza H1N1, VIRUS_Influenza H3, VIRUS_Influenza B, VIRUS_Metapneumovírus, VIRUS_Sincicial A, VIRUS_Sincicial B, VIRUS_Rinovírus, VIRUS_Parainfluenza 1, VIRUS_Parainfluenza 2, VIRUS_Parainfluenza 3, VIRUS_Parainfluenza 4